In [ ]:
%pip install numpy
%pip install plyfile

# Ordre Morton

In [ ]:
##Morton Code
from plyfile import PlyElement, PlyData
import numpy as np

ply = PlyData.read("Avio.ply")

v = ply['vertex']

# Extract as NumPy arrays
points = np.vstack((v['x'], v['y'], v['z'])).T
colors = np.vstack((v['red'], v['green'], v['blue'])).T.astype(np.uint8)

print(points.shape, colors.shape)

def normalize_to_int(points, bits=10):
    max_val = (1 << bits) - 1
    pts = points - points.min(axis=0)
    pts = pts / pts.max(axis=0)
    return (pts * max_val).astype(np.uint32)

def part1by2(n):
    n &= 0x3ff
    n = (n | n << 16) & 0x30000ff
    n = (n | n << 8)  & 0x300f00f
    n = (n | n << 4)  & 0x30c30c3
    n = (n | n << 2)  & 0x9249249
    return n

def morton3D(x, y, z):
    return (part1by2(z) << 2) | (part1by2(y) << 1) | part1by2(x)

int_pts = normalize_to_int(points, bits=10)

morton = np.array([
    morton3D(x, y, z) for x, y, z in int_pts
], dtype=np.uint64)

order = np.argsort(morton)

points_sorted = points[order]
colors_sorted = colors[order]

# Build structured array for vertices
vertex_data = np.empty(points_sorted.shape[0],
                       dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4'),
                              ('red', 'u1'), ('green', 'u1'), ('blue', 'u1')])

vertex_data['x'] = points_sorted[:,0]
vertex_data['y'] = points_sorted[:,1]
vertex_data['z'] = points_sorted[:,2]
vertex_data['red']   = colors_sorted[:,0]
vertex_data['green'] = colors_sorted[:,1]
vertex_data['blue']  = colors_sorted[:,2]

vertex_el = PlyElement.describe(vertex_data, 'vertex')

# Write output
out = PlyData([vertex_el], text=False)
out.write("Avio Morton.ply")

# Ordre Aleatori

In [ ]:
##Shuffle
from plyfile import PlyData, PlyElement
import numpy as np

ply = PlyData.read("Avio.ply")
vertex = ply['vertex']

# Extract all properties as a structured array
data = vertex.data
data = np.array(data)

# Shuffle rows
np.random.shuffle(data)

# Write shuffled PLY
el = PlyElement.describe(data, 'vertex')
PlyData([el], text=ply.text).write("Avio Shuffle.ply")

# Ordre Morton Shuffle

In [ ]:
##Morton Shuffle
from plyfile import PlyElement, PlyData
import numpy as np
import random

ply = PlyData.read("Avio.ply")

v = ply['vertex']

# Extract as NumPy arrays
points = np.vstack((v['x'], v['y'], v['z'])).T
colors = np.vstack((v['red'], v['green'], v['blue'])).T.astype(np.uint8)

print(points.shape, colors.shape)

def normalize_to_int(points, bits=10):
    max_val = (1 << bits) - 1
    pts = points - points.min(axis=0)
    pts = pts / pts.max(axis=0)
    return (pts * max_val).astype(np.uint32)

def part1by2(n):
    n &= 0x3ff
    n = (n | n << 16) & 0x30000ff
    n = (n | n << 8)  & 0x300f00f
    n = (n | n << 4)  & 0x30c30c3
    n = (n | n << 2)  & 0x9249249
    return n

def morton3D(x, y, z):
    return (part1by2(z) << 2) | (part1by2(y) << 1) | part1by2(x)

int_pts = normalize_to_int(points, bits=10)

morton = np.array([
    morton3D(x, y, z) for x, y, z in int_pts
], dtype=np.uint64)

order = np.argsort(morton)

points_sorted = points[order]
colors_sorted = colors[order]

batch_size = 128
N = points_sorted.shape[0]
num_batches = (N + batch_size - 1) // batch_size 

points_batches = [
    points_sorted[i*batch_size:(i+1)*batch_size]
    for i in range(num_batches)
]

colors_batches = [
    colors_sorted[i*batch_size:(i+1)*batch_size]
    for i in range(num_batches)
]

indices = list(range(num_batches))
random.shuffle(indices)

points_shuffled = [points_batches[i] for i in indices]
colors_shuffled = [colors_batches[i] for i in indices]

points_final = np.vstack(points_shuffled)
colors_final = np.vstack(colors_shuffled)

# Build structured array for vertices
vertex_data = np.empty(points_sorted.shape[0],
                       dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4'),
                              ('red', 'u1'), ('green', 'u1'), ('blue', 'u1')])

vertex_data['x'] = points_final[:,0]
vertex_data['y'] = points_final[:,1]
vertex_data['z'] = points_final[:,2]
vertex_data['red']   = colors_final[:,0]
vertex_data['green'] = colors_final[:,1]
vertex_data['blue']  = colors_final[:,2]

vertex_el = PlyElement.describe(vertex_data, 'vertex')

# Write output
out = PlyData([vertex_el], text=False)
out.write("Avio Morton Shuffle.ply")

# Localitat

In [ ]:
from plyfile import PlyElement, PlyData
import numpy as np
import matplotlib.pyplot as plt

# Read the PLY file
ply = PlyData.read("Manuscript Morton.ply")

# Extract the vertex data
v = ply['vertex']

# Stack x, y, z coordinates into a single NumPy array (Nx3)
points = np.vstack((v['x'], v['y'], v['z'])).T

# Calculate the coordinate distances between consecutive points using Euclidean distance
coord_distances = np.sqrt(np.sum(np.diff(points, axis=0)**2, axis=1))

# Calculate the mean distance
mean_distance = np.mean(coord_distances)

# Print the result
print(f"Mean distance: {mean_distance}")

# --- 3D Plotting ---
'''
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot the 3D scatter plot of the points (x, y, z)
ax.scatter(points[1:, 0], points[1:, 1], points[1:, 2], c='b', marker='o', label='Points')

# Optional: Connect consecutive points with lines to show distances
for i in range(1, len(points)):
    ax.plot(
        [points[i-1, 0], points[i, 0]],
        [points[i-1, 1], points[i, 1]],
        [points[i-1, 2], points[i, 2]],
        color='r', linestyle='-', linewidth=0.5  # Red lines connecting consecutive points
    )

# Labels and title
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('3D Plot of Points and Distances')

# Show the plot
plt.show()
'''
# --- 2D Line Plot for Distance Progression ---
fig_2d = plt.figure(figsize=(12, 8))

# Create a 2D axis for the second plot
ax_2d = fig_2d.add_subplot(111)

# Plot the 2D line plot of distances (ignoring the first point)
ax_2d.plot(range(1, len(coord_distances)+1), coord_distances, color='r', label='Distance Progression')

# Labels and title for the 2D plot
ax_2d.set_xlabel('Index')
ax_2d.set_ylabel('Distance')
ax_2d.set_title('Distance Progression Between Consecutive Points')
ax_2d.grid(True)

# Display the 2D plot
plt.show()

# Gràfics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def lebesgue2d(num_points):
    # Function to calculate Lebesgue coordinates
    def lebesgue_coords(z):
        coords = {'x': 0, 'y': 0}
        shift_mask = 0x55555555  # Mask for even bits
        
        # Mask out even bits for x, odd bits for y
        coords['x'] = z & shift_mask
        coords['y'] = (z & 0xaaaaaaaa) >> 1  # Shift the odd bits to y
        
        shift_mask = 0xfffffffc  # Adjust the mask to handle 2-bit wide groups
        for i in range(32):
            # Compress bits by extracting the top bits and shifting them down one
            x_upper = (coords['x'] & shift_mask) >> 1
            y_upper = (coords['y'] & shift_mask) >> 1
            
            # Clear out the top bits from x and reintroduce the shifted bits
            coords['x'] = x_upper | (coords['x'] & ~shift_mask)
            coords['y'] = y_upper | (coords['y'] & ~shift_mask)
            
            shift_mask <<= 1  # Reduce the mask size, processing pairs of bits
        
        return coords
    
    # Function to compute the Lebesgue coordinates for a range of z values
    def lebesgue(z_values):
        x_values = []
        y_values = []
        
        for z in z_values:
            coords = lebesgue_coords(z)
            x_values.append(coords['x'])
            y_values.append(coords['y'])
        
        return pd.DataFrame({'x': x_values, 'y': y_values})
    
    z_values = range(0, num_points)  
    lebesgue_points = pd.DataFrame({'z': z_values})
    
    # Apply the lebesgue function to calculate 'x' and 'y' coordinates
    lebesgue_points[['x', 'y']] = lebesgue(z_values)
    
    # Print the resulting DataFrame
    print(lebesgue_points)
    
    # Plot the data with lines connecting the points
    plt.figure(figsize=(6, 6))
    
    # Plot the 'x' and 'y' points as a line
    plt.plot(lebesgue_points['x'], lebesgue_points['y'], marker='o', color='b', linestyle='-', markersize=4)
    
    # Adding labels and title
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Lebesgue Curve: Connecting Points')
    
    # Show grid for better readability
    plt.grid(True)
    
    # Display the plot
    plt.show()
    
    # Calculate the coordinate distances
    lebesgue_points['coord_distance'] = np.sqrt(
        (lebesgue_points['x'] - lebesgue_points['x'].shift(1))**2 + 
        (lebesgue_points['y'] - lebesgue_points['y'].shift(1))**2
    )
    
    # Remove rows with NaN values (first row will have NaN distance)
    lebesgue_locality = lebesgue_points.dropna(subset=['coord_distance'])
    
    # Calculate the mean distance
    mean_distance = lebesgue_locality['coord_distance'].mean()
    
    # Print the result
    print(f"Mean distance: {mean_distance}")
    
    lebesgue_locality = lebesgue_points.dropna(subset=['coord_distance'])
    
    # Plot the coordinate distances
    plt.figure(figsize=(12, 6))
    plt.plot(lebesgue_locality['z'], lebesgue_locality['coord_distance'], marker='o', color='b', linestyle='-', markersize=4)
    plt.xlabel('z')
    plt.ylabel('Distance Between Points')
    plt.title('Distance Between Consecutive Points')
    plt.grid(True)
    plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def lebesgue3d(num_points):
    # Function to calculate Lebesgue coordinates
    def lebesgue_coords(w):
        coords = {'x': 0, 'y': 0, 'z':0}
        
        
        coords['x'] = w & 0x49249249
        coords['y'] = (w & 0x92492492) >> 1
        coords['z'] = (w & 0x24924924) >> 1
        
        shift_mask = 0xfffffff8  # Adjust the mask to handle 2-bit wide groups
        for i in range(32):
            # Compress bits by extracting the top bits and shifting them down one
            x_upper = (coords['x'] & shift_mask) >> 1
            y_upper = (coords['y'] & shift_mask) >> 1
            z_upper = (coords['z'] & shift_mask) >> 1
            
            # Clear out the top bits from x and reintroduce the shifted bits
            coords['x'] = x_upper | (coords['x'] & ~shift_mask)
            coords['y'] = y_upper | (coords['y'] & ~shift_mask)
            coords['z'] = z_upper | (coords['z'] & ~shift_mask)
            
            shift_mask <<= 1  # Reduce the mask size, processing pairs of bits
        
        return coords
    
    # Function to compute the Lebesgue coordinates for a range of z values
    def lebesgue(w_values):
        x_values = []
        y_values = []
        z_values = []
        
        for w in w_values:
            coords = lebesgue_coords(w)
            x_values.append(coords['x'])
            y_values.append(coords['y'])
            z_values.append(coords['z'])
        
        return pd.DataFrame({'x': x_values, 'y': y_values, 'z': z_values})
    
    w_values = range(0, num_points)  
    lebesgue_points = pd.DataFrame({'w': w_values})
    
    # Apply the lebesgue function to calculate 'x' and 'y' coordinates
    lebesgue_points[['x', 'y', 'z']] = lebesgue(w_values)
    
    # Print the resulting DataFrame
    print(lebesgue_points)

    # Calculate the coordinate distances
    lebesgue_points['coord_distance'] = np.sqrt(
        (lebesgue_points['x'] - lebesgue_points['x'].shift(1))**2 + 
        (lebesgue_points['y'] - lebesgue_points['y'].shift(1))**2 + 
        (lebesgue_points['z'] - lebesgue_points['z'].shift(1))**2
    )
    
    # Remove rows with NaN values (first row will have NaN distance)
    lebesgue_locality = lebesgue_points.dropna(subset=['coord_distance'])
    
    # Calculate the mean distance
    mean_distance = lebesgue_locality['coord_distance'].mean()
    
    # Print the result
    print(f"Mean distance: {mean_distance}")
    
    lebesgue_locality = lebesgue_points.dropna(subset=['coord_distance'])
    
    # 3D Plotting
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot the 3D scatter plot of the points (x, y, z)
    ax.scatter(lebesgue_locality['x'], lebesgue_locality['y'], lebesgue_locality['z'], c='b', marker='o', label='Points')
    
    # Optional: Connect consecutive points with lines to show distances
    for i in range(1, len(lebesgue_locality)):
        ax.plot(
            [lebesgue_locality['x'].iloc[i-1], lebesgue_locality['x'].iloc[i]],
            [lebesgue_locality['y'].iloc[i-1], lebesgue_locality['y'].iloc[i]],
            [lebesgue_locality['z'].iloc[i-1], lebesgue_locality['z'].iloc[i]],
            color='r', linestyle='-', linewidth=0.5  # Red lines connecting consecutive points
        )
    
    # Labels and title
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title('3D Plot of Lebesgue Points')
    
    # Show the plot
    plt.show()
    
    # --- 2. 2D Line Plot for Distance Progression (Separate Figure) ---
    fig_2d = plt.figure(figsize=(12, 8))
    
    # Create a 2D axis for the second plot
    ax_2d = fig_2d.add_subplot(111)
    
    # Plot the 2D line plot of distances
    ax_2d.plot(lebesgue_locality['w'], lebesgue_locality['coord_distance'], color='r', label='Distance Progression')
    
    # Labels and title for the 2D plot
    ax_2d.set_xlabel('w')
    ax_2d.set_ylabel('Distance')
    ax_2d.set_title('2D Plot of Distance Progression')
    ax_2d.grid(True)
    
    # Display the 2D plot
    plt.show()

In [ ]:
lebesgue3d(4096)

# Gràfic Goat

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data
variants = ['Original', 'Morton', 'Shuffle', 'Shuffle Morton']
conditions = ['GL_Points/No Depth Test', 'GL_points/Depth Test', 'Compute Shader Z Test', 'Compute Shader No Z Test']

fps_data = {
    'Original': [342.827, 343.32, 356.735, 360.297],
    'Morton': [338.133, 334.999, 356.35, 355.693],
    'Shuffle': [334.094, 341.704, 344.852, 346.58],
    'Shuffle Morton': [343.548, 338.388, 359.554, 364.79]
}

x = np.arange(len(variants)) # the label locations
width = 0.13 # the width of the bars


plt.figure(figsize=(12, 6))


for i, condition in enumerate(conditions):
    condition_values = [1000/fps_data[variant][i] for variant in variants]
    plt.bar(x + i*width, condition_values, width, label=condition)


plt.title('FPS Comparison Across Different Orders and Shaders')
plt.ylabel('milliseconds (ms)')
plt.xticks(x + width*2.5, variants)
plt.axhline(1000/fps_data['Original'][2], linestyle='--', linewidth=1)
plt.legend(loc='upper left', bbox_to_anchor=(1,1)) # Move legend outside the plot
plt.tight_layout()
plt.show()

# Gràfic Gerro

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data
variants = ['Original', 'Morton', 'Shuffle', 'Shuffle Morton']
conditions = ['GL_Points/No Depth Test', 'GL_points/Depth Test', 'Compute Shader Z Test', 'Compute Shader No Z Test']

fps_data = {
    'Original': [355.088, 342.665, 355.837, 360.122],
    'Morton': [350.347, 343.965, 352.58, 356.307],
    'Shuffle': [347.222, 343.777, 342.467, 343.976],
    'Shuffle Morton': [349.927, 345.038, 356.333, 361.486]
}

x = np.arange(len(variants)) # the label locations
width = 0.13 # the width of the bars


plt.figure(figsize=(12, 6))


for i, condition in enumerate(conditions):
    condition_values = [1000/fps_data[variant][i] for variant in variants]
    plt.bar(x + i*width, condition_values, width, label=condition)


plt.title('FPS Comparison Across Different Orders and Shaders')
plt.ylabel('milliseconds (ms)')
plt.xticks(x + width*2.5, variants)
plt.axhline(1000/fps_data['Original'][2], linestyle='--', linewidth=1)
plt.legend(loc='upper left', bbox_to_anchor=(1,1)) # Move legend outside the plot
plt.tight_layout()
plt.show()

# Gràfic Pàgina

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data
variants = ['Original', 'Morton', 'Shuffle', 'Shuffle Morton']
conditions = ['GL_Points/No Depth Test', 'GL_points/Depth Test', 'Compute Shader Z Test', 'Compute Shader No Z Test']

fps_data = {
    'Original': [380.086, 351.809, 379.585, 408.383],
    'Morton': [378.057, 348.89, 381.391, 409.221],
    'Shuffle': [382.317, 384.651, 356.432, 304.527],
    'Shuffle Morton': [388.805, 364.99, 393.398, 417.226]
}

x = np.arange(len(variants)) # the label locations
width = 0.13 # the width of the bars


plt.figure(figsize=(12, 6))


for i, condition in enumerate(conditions):
    condition_values = [1000/fps_data[variant][i] for variant in variants]
    plt.bar(x + i*width, condition_values, width, label=condition)


plt.title('FPS Comparison Across Different Orders and Shaders')
plt.ylabel('milliseconds (ms)')
plt.xticks(x + width*2.5, variants)
plt.axhline(1000/fps_data['Original'][2], linestyle='--', linewidth=1)
plt.legend(loc='upper left', bbox_to_anchor=(1,1)) # Move legend outside the plot
plt.tight_layout()
plt.show()

# Gràfic Avió

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data
variants = ['Original', 'Morton', 'Shuffle', 'Shuffle Morton']
conditions = ['GL_Points/No Depth Test', 'GL_points/Depth Test', 'Compute Shader Z Test', 'Compute Shader No Z Test']

fps_data = {
    'Original': [348.596, 341.081, 341.869, 338.476],
    'Morton': [347.064, 342.999, 341.654, 347.724],
    'Shuffle': [349.405, 346.015, 341.25, 347.115],
    'Shuffle Morton': [357.354, 359.872, 338.222, 341.39]
}

x = np.arange(len(variants)) # the label locations
width = 0.13 # the width of the bars


plt.figure(figsize=(12, 6))


for i, condition in enumerate(conditions):
    condition_values = [1000/fps_data[variant][i] for variant in variants]
    plt.bar(x + i*width, condition_values, width, label=condition)


plt.title('FPS Comparison Across Different Orders and Shaders')
plt.ylabel('milliseconds (ms)')
plt.xticks(x + width*2.5, variants)
plt.axhline(1000/fps_data['Original'][1], linestyle='--', linewidth=1)
plt.legend(loc='upper left', bbox_to_anchor=(1,1)) # Move legend outside the plot
plt.tight_layout()
plt.show()

# Comparació amb el Gerard Perelló

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data
variants = ['Original']
conditions = ['GL_Points/No Depth Test', 'GL_points/Depth Test', 'Compute Shader Z Test', 'Compute Shader No Z Test',
             'Gerard']

fps_data = {
    'Original': [73.1, 72.709, 42.782, 43.725, 22.9]
}

x = np.arange(len(variants)) # the label locations
width = 0.13 # the width of the bars


plt.figure(figsize=(12, 6))


for i, condition in enumerate(conditions):
    condition_values = [fps_data[variant][i] for variant in variants]
    plt.bar(x + i*width, condition_values, width, label=condition)


plt.title('FPS Comparison Across Different Shaders')
plt.ylabel('milliseconds (ms)')
plt.xticks(x + width*2.5, variants)
plt.legend(loc='upper left', bbox_to_anchor=(1,1)) # Move legend outside the plot
plt.tight_layout()
plt.show()

# Modificar Point Clouds

In [ ]:
from plyfile import PlyElement, PlyData
import numpy as np

ply = PlyData.read("lucy.ply")

v = ply['vertex']

# Extract as NumPy arrays
points = np.vstack((v['x'], v['y'], v['z'])).T

vertex_data = np.empty(points.shape[0],
                       dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4'),
                              ('red', 'u1'), ('green', 'u1'), ('blue', 'u1')])

vertex_data['x'] = points[:,0]
vertex_data['y'] = points[:,1]
vertex_data['z'] = points[:,2]
vertex_data['red'][:] = 255
vertex_data['green'][:] = 255
vertex_data['blue'][:] = 255

vertex_el = PlyElement.describe(vertex_data, 'vertex')

# Write output
out = PlyData([vertex_el], text=False)
out.write("Lucy_color.ply")

# Diagrama de Gantt

In [ ]:
import matplotlib.pyplot as plt
import datetime as dt

# Define tasks and periods (month-based)
tasks = [
    ("Recerca", dt.date(2025, 9, 17), dt.date(2025, 12, 21)),
    ("Disseny", dt.date(2025, 10, 2), dt.date(2025, 11, 5)),
    ("Codificació", dt.date(2025, 10, 27), dt.date(2026, 1, 10)),
    ("Redacció", dt.date(2025, 12, 9), dt.date(2026, 1, 14)),
]

months = [
    dt.date(2025, 9, 1),
    dt.date(2025, 10, 1),
    dt.date(2025, 11, 1),
    dt.date(2025, 12, 1),
    dt.date(2026, 1, 1),
    dt.date(2026, 2, 1),
]

month_labels = ["Setembre", "Octubre", "Novembre", "Desembre", "Gener", "Febrer"]

# Plot
plt.figure(figsize=(10, 3))
y_pos = range(len(tasks))

for i, (task, start, end) in enumerate(tasks):
    plt.barh(i, (end - start).days, left=start)

plt.yticks(y_pos, [t[0] for t in tasks])
plt.xticks(months, month_labels)
plt.xlabel("Temps")
plt.title("Diagrama de Gantt")
plt.tight_layout()
plt.show()